In [ ]:
# Сохраняем эксель файлы в нормальном виде в dc2

import pandas as pd
from pathlib import Path

# Пути
old_root = Path("Digital_core")
new_root = Path("Digital_core_v2")

# Создаём новую папку
new_root.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("КОНВЕРТАЦИЯ EXCEL → CSV")
print("=" * 60)

# Проходим по всем папкам в Digital core
for old_field_dir in old_root.iterdir():
    if not old_field_dir.is_dir():
        continue
    
    print(f"\nОбработка: {old_field_dir.name}")
    
    # Ищем Excel файл
    excel_files = [f for f in old_field_dir.glob("*.xlsx") if not f.name.startswith('~$')]
    
    if not excel_files:
        print(f"  ⚠️ Excel не найден, пропускаем")
        continue
    
    excel_path = excel_files[0]
    excel_name = excel_path.stem
    
    try:
        # Читаем как в твоей функции
        data = pd.read_excel(excel_path, usecols='B:G', skiprows=8, header=None)
        data = data.drop(5, axis=1)
        for col in [1, 3, 4]:
            data[col] = pd.to_numeric(data[col], errors='coerce')
        data = data.dropna(subset=[1, 3, 4])
        data[2] = data[1] + data[4]
        data[1] = data[1] + data[3]
        data = data.drop([3, 4], axis=1)
        data = data.dropna()
        data = data.rename(columns={1: 'depth_from', 2: 'depth_to', 6: 'mineral'})
        
        # Создаём папку в Digital_core_v2
        new_field_dir = new_root / old_field_dir.name
        new_field_dir.mkdir(parents=True, exist_ok=True)
        
        # Сохраняем CSV с оригинальным именем
        csv_path = new_field_dir / f"{excel_name}.csv"
        data.to_csv(csv_path, index=False, encoding='utf-8-sig')
        
        print(f"  ✅ Сохранено: {csv_path.name}")
        print(f"     Строк: {len(data)}")
        
    except Exception as e:
        print(f"  ❌ Ошибка: {e}")

print("\n" + "=" * 60)
print("ГОТОВО!")
print("=" * 60)

In [ ]:
# сохраняем все остальные файлы то есть фотографии
import pandas as pd
import cv2
import json
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image

original_root = Path("Digital_core")
new_root = Path("Digital_core_v2")
METERS_TO_CM = 100

def load_image(path):
    with Image.open(path) as img:
        return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

total_crops = 0

for old_field_dir in original_root.iterdir():
    if not old_field_dir.is_dir():
        continue
    
    photo_dir = old_field_dir / "ФОТО"
    if not photo_dir.exists():
        continue
    
    new_field_dir = new_root / old_field_dir.name
    csv_files = list(new_field_dir.glob("*.csv"))
    if not csv_files:
        continue
    
    df = pd.read_csv(csv_files[0], encoding='utf-8-sig')
    
    all_metadata = []
    field_crops = 0
    counter = 0
    
    for light_type in ["ДС", "УФ"]:
        old_light_dir = photo_dir / light_type
        if not old_light_dir.exists():
            continue
        
        new_light_dir = new_field_dir / "ФОТО" / light_type
        new_light_dir.mkdir(parents=True, exist_ok=True)
        
        for img_path in tqdm(list(old_light_dir.glob("*.jpeg")), desc=f"{old_field_dir.name} - {light_type}"):
            stem = img_path.stem
            try:
                underscore_pos = stem.rfind('_')
                dash_pos = stem.rfind('-')
                depth_from = float(stem[underscore_pos + 1:dash_pos - 1])
                depth_to = float(stem[dash_pos + 2:])
            except:
                continue
            
            img = load_image(img_path)
            height = img.shape[0]
            depth_range_m = depth_to - depth_from
            depth_range_cm = depth_range_m * METERS_TO_CM
            depth_scale = height / depth_range_cm
            
            overlapping = df[(df['depth_from'] <= depth_to) & (df['depth_to'] >= depth_from)]
            
            for _, row in overlapping.iterrows():
                mineral_from = row['depth_from']
                mineral_to = row['depth_to']
                
                intersect_start = max(mineral_from, depth_from)
                intersect_end = min(mineral_to, depth_to)
                
                if intersect_start >= intersect_end:
                    continue
                
                y_start = round((intersect_start - depth_from) * METERS_TO_CM * depth_scale)
                y_end = round((intersect_end - depth_from) * METERS_TO_CM * depth_scale)
                y_start = max(0, y_start)
                y_end = min(height, y_end)
                
                if y_start >= y_end:
                    continue
                
                cropped = img[y_start:y_end, :, :]
                filename = f"{img_path.stem}_crop{counter:04d}_{row['mineral']}.jpg"
                filename = filename.replace(' ', '_').replace('/', '_').replace('?', '')
                
                Image.fromarray(cropped).save(str(new_light_dir / filename))
                
                # Сохраняем метаданные как в старом коде
                all_metadata.append({
                    'id': len(all_metadata),
                    'mineral': row['mineral'],
                    'depth_range': [intersect_start, intersect_end],
                    'original_image': str(img_path),
                    'has_continuation_up': mineral_from < depth_from,
                    'has_continuation_down': mineral_to > depth_to,
                    'original_depth_from': mineral_from,
                    'original_depth_to': mineral_to,
                    'image_filename': filename,
                    'light_type': light_type
                })
                field_crops += 1
                total_crops += 1
                counter += 1
    
    if all_metadata:
        meta_path = new_field_dir / "cropped_metadata.json"
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(all_metadata, f, ensure_ascii=False, indent=2)
        print(f"  ✅ {old_field_dir.name}: {field_crops} кусков")

print(f"\n✅ ВСЕГО: {total_crops} кусков")

# Проверка
ds_count = sum(1 for _ in new_root.rglob("ДС/*.jpg"))
uf_count = sum(1 for _ in new_root.rglob("УФ/*.jpg"))
print(f"ДС: {ds_count} файлов, УФ: {uf_count} файлов")

In [ ]:
# склейка разрозненных кусков в цельные изображения и последующее разбиение на части если склеенное изображение
# слишком велико

import json
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from PIL import Image

v2_root = Path("Digital_core_v2")
v3_root = Path("Digital_core_v3")
v3_root.mkdir(exist_ok=True)

MAX_HEIGHT_PX = 25000  # максимальная высота одного изображения

def load_image(path):
    with Image.open(path) as img:
        return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def save_image(path, img):
    img = np.clip(img, 0, 255).astype(np.uint8)
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    elif img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    Image.fromarray(img).save(str(path), quality=90)

def merge_and_split(parts, d_from, d_to):
    """
    Склеивает части, при необходимости разбивает на несколько изображений,
    возвращает список кортежей (изображение, глубина_начала, глубина_конца)
    """
    images = [load_image(p['path']) for p in parts]
    max_width = max(img.shape[1] for img in images)
    
    # Выравниваем ширину
    aligned = []
    for img in images:
        h, w = img.shape[:2]
        if w < max_width:
            pad = max_width - w
            aligned.append(cv2.copyMakeBorder(img, 0, 0, 0, pad, cv2.BORDER_CONSTANT, value=[0,0,0]))
        else:
            aligned.append(img)
    
    # Склеиваем всё в одно изображение
    full = np.vstack(aligned)
    full_height = full.shape[0]
    
    # Если высота в пределах лимита, возвращаем один фрагмент
    if full_height <= MAX_HEIGHT_PX:
        return [(full, d_from, d_to)]
    
    # Иначе разбиваем на части пропорционально высоте
    total_length = d_to - d_from
    parts_count = (full_height + MAX_HEIGHT_PX - 1) // MAX_HEIGHT_PX
    chunks = []
    for i in range(parts_count):
        y_start = i * MAX_HEIGHT_PX
        y_end = min((i + 1) * MAX_HEIGHT_PX, full_height)
        chunk = full[y_start:y_end, :]
        # Реальные глубины для этой части
        part_height = y_end - y_start
        part_frac_start = y_start / full_height
        part_frac_end   = (y_start + part_height) / full_height
        part_d_from = d_from + total_length * part_frac_start
        part_d_to   = d_from + total_length * part_frac_end
        chunks.append((chunk, part_d_from, part_d_to))
    return chunks

total_files = 0

for field_dir in tqdm(list(v2_root.iterdir()), desc="Обработка месторождений"):
    if not field_dir.is_dir():
        continue
    meta_path = field_dir / "cropped_metadata.json"
    if not meta_path.exists():
        continue
    with open(meta_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    groups = defaultdict(list)
    for item in metadata:
        key = (item['mineral'], item['original_depth_from'], item['original_depth_to'], item['light_type'])
        groups[key].append(item)
    
    for light in ["ДС", "УФ"]:
        light_groups = {k: v for k, v in groups.items() if k[3] == light}
        if not light_groups:
            continue
        dst_light_dir = v3_root / field_dir.name / light
        dst_light_dir.mkdir(parents=True, exist_ok=True)
        
        for (mineral, d_from, d_to, _), items in tqdm(light_groups.items(), desc=f"{field_dir.name} {light}"):
            items_sorted = sorted(items, key=lambda x: x['depth_range'][0])
            parts_with_paths = []
            for it in items_sorted:
                img_path = field_dir / "ФОТО" / light / it['image_filename']
                if img_path.exists():
                    parts_with_paths.append({'path': img_path})
            if not parts_with_paths:
                continue
            # Получаем склеенные/разделённые изображения с корректными интервалами
            chunks = merge_and_split(parts_with_paths, d_from, d_to)
            for chunk_img, chunk_from, chunk_to in chunks:
                # Имя файла: минерал_начало_конец.jpg (без _part)
                filename = f"{mineral}_{chunk_from:.3f}_{chunk_to:.3f}.jpg"
                filename = filename.replace(' ', '_').replace('/', '_')
                dst_path = dst_light_dir / filename
                save_image(dst_path, chunk_img)
                total_files += 1

print(f"\n✅ Создано файлов в Digital_core_v3: {total_files}")

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import re

v3_root = Path("Digital_core_v3")
v4_root = Path("Digital_core_v4")
v4_root.mkdir(exist_ok=True)

TILE_CM = 5          # 5 см
OVERLAP_CM = 1.0     # 1 см перекрытие
METERS_TO_CM = 100

def load_image(path):
    with Image.open(path) as img:
        return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def save_image(path, img):
    if img is None or img.size == 0:
        return False
    img = np.clip(img, 0, 255).astype(np.uint8)
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    elif img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    Image.fromarray(img).save(str(path), quality=90)
    return True

def split_mineral(img, depth_from, depth_to):
    h, w = img.shape[:2]
    total_length_m = depth_to - depth_from
    if total_length_m <= 0 or h <= 0:
        return []
    px_per_cm = h / (total_length_m * METERS_TO_CM)
    step_cm = TILE_CM - OVERLAP_CM
    step_px = int(step_cm * px_per_cm)
    tile_px = int(TILE_CM * px_per_cm)
    if tile_px <= 0:
        return []   # теоретически при корректных данных не должно быть
    fragments = []
    current_cm = 0.0
    while current_cm + TILE_CM <= total_length_m * METERS_TO_CM + 1e-6:  # допуск
        y_start = int(round(current_cm * px_per_cm))
        y_end = y_start + tile_px
        if y_end > h:
            break
        fragment = img[y_start:y_end, :]
        if fragment.shape[0] == 0:
            break
        frag_depth_from = depth_from + current_cm / METERS_TO_CM
        frag_depth_to = frag_depth_from + TILE_CM / METERS_TO_CM
        fragments.append((fragment, frag_depth_from, frag_depth_to))
        current_cm += step_cm
    return fragments

total_frags = 0

for field_dir in tqdm(list(v3_root.iterdir()), desc="Обработка месторождений"):
    if not field_dir.is_dir():
        continue
    for light in ["ДС", "УФ"]:
        src_dir = field_dir / light
        if not src_dir.exists():
            continue
        dst_dir = v4_root / field_dir.name / light
        dst_dir.mkdir(parents=True, exist_ok=True)
        img_paths = list(src_dir.glob("*.jpg"))
        if not img_paths:
            continue
        for img_path in tqdm(img_paths, desc=f"{field_dir.name}/{light}", leave=False):
            # Парсим глубины из имени файла (два последних числа)
            stem = img_path.stem
            # Ищем два числа с плавающей точкой в конце строки
            match = re.search(r'(\d+\.\d+)_(\d+\.\d+)$', stem)
            if not match:
                # возможно, формат иной – попробуем другой
                match = re.search(r'(\d+\.\d+)_(\d+\.\d+)\.jpg', img_path.name)
            if not match:
                continue
            depth_from = float(match.group(1))
            depth_to   = float(match.group(2))
            img = load_image(img_path)
            if img is None:
                continue
            fragments = split_mineral(img, depth_from, depth_to)
            for i, (frag_img, fd_from, fd_to) in enumerate(fragments):
                frag_name = f"{img_path.stem}_frag{i:04d}.jpg"
                frag_path = dst_dir / frag_name
                if save_image(frag_path, frag_img):
                    total_frags += 1

print(f"\n✅ Создано фрагментов в Digital_core_v4: {total_frags}")